In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from langchain_community.cache import RedisCache
from langchain_core.globals import set_llm_cache
from redis import Redis
import os
from backend.async.async_runner import LLMRunner

app = FastAPI(title="LLM Multi-Runner")
runner = LLMRunner()

# 1. Redis 캐시 설정 (LangChain 내장 기능 활용)
# 이렇게 설정하면 model.invoke() 시 자동으로 캐시를 확인합니다.
redis_client = Redis.from_url(os.getenv("REDIS_URL", "redis://localhost:6379/0"))
set_llm_cache(RedisCache(redis_=redis_client))

class QueryRequest(BaseModel):
    query: str
    use_cache: bool = True

@app.post("/benchmark")
async def benchmark_llms(request: QueryRequest):
    if not request.query:
        raise HTTPException(status_code=400, detail="Query is empty")

    # 2. 실행 (캐싱은 LangChain 내부에서 자동 처리되거나, 
    # Semantic Cache를 원하면 여기서 별도 로직 구현 가능)
    results = await runner.run_all(request.query)

    # 3. 통계 집계
    total_cost = sum(r["cost_usd"] for r in results if r["status"] == "success")
    fastest_model = min(
        [r for r in results if r["status"] == "success"], 
        key=lambda x: x["latency"], 
        default=None
    )

    return {
        "query": request.query,
        "results": results,
        "summary": {
            "total_cost_usd": total_cost,
            "fastest_model": fastest_model["model"] if fastest_model else None,
            "model_count": len(results)
        }
    }

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)